# **Modelo LightGCN**
### Proyecto Hito 2
### Sistemas Recomendadores IIC3633-1 2025-2
### **Grupo 3:** 

- Nicolás Antonio Bueno Abett de la Torre 

- Felipe Andrés Fuentes González

- Jorge Andrés Jacque Palma

- Francisco Nicolás Solís Gormaz

## Índice

>[0- Instalación de librerías](#0--instalación-de-librerías)

>[1- Carga de datos](#1--carga-de-datos)

>[2- Definición del modelo, formateo de datos, y entrenamiento](#2--definición-del-modelo-formateo-de-datos-y-entrenamiento)

>[3- Generación de recomendaciones](#3--generación-de-recomendaciones)

>[4- Métricas](#4--métricas)

>[5- Referencias](#5--referencias)

## 0- Instalación de librerías

En caso de usar Colab, correr estas celdas. Si se corre en local, se deben tener exactamente las mismas versiones de las librerías indicadas en estas celdas. Las versiones de las librerías son:

- numpy: 1.25.0
- pandas: 2.2.2
- scipy: 1.10.1
- tqdm: 4.66.6
- torch: 2.5.1+cpu
- recbole: 1.2.1

In [ ]:
# !pip uninstall -y numpy
# !pip install numpy==1.25

In [ ]:
# !pip uninstall -y pandas
# !pip install numpy==2.2.2

In [ ]:
# !pip uninstall -y scipy
# !pip install scipy==1.10.1

In [ ]:
# !pip uninstall -y tqdm
# !pip install tqdm==4.66.6

In [ ]:
# !pip uninstall -y torch
# !pip uninstall -y torchvision
# !pip uninstall -y torchaudio
# !pip install torch==2.5.1+cpu --index-url https://download.pytorch.org/whl/cpu

In [ ]:
# !pip install recbole==1.2.1

## 1- Carga de datos

Se leen los archivos de datos y se almacenan en un dataframe. El primero, df_original, lee el archivo original del dataset. El segundo, "df_con_ids", lee el archivo del dataset modificado con ids de usuario e ítem agregadas, tal como se hizo en el Hito 1 del proyecto al implementar los modelos referenciales (es el mismo archivo creado en /modelos_ref/modelos_ref.ipynb). El tercero, "df_final", es el mismo contenido de "df_con_ids", pero tomando sólo las columnas de id de usuario, id de ítem, y rating, con estos últimos convertidos a escala del 1 al 5, tal como se hizo en el Hito 1 del proyecto al implementar los modelos referenciales (es el mismo archivo creado en /modelos_ref/modelos_ref.ipynb). Los dataframes son:

In [1]:
import pandas as pd

df_original = pd.read_csv('video_game_reviews.csv')
df_con_ids = pd.read_csv('video_game_reviews_with_userid.csv')
df_final = pd.read_csv('video_game_reviews_with_userid_clean.csv')

En este diccionario "info_videojuegos" se guarda la id del videojuego junto a su título correspondiente, para luego obtener información de este (como su género, plataforma, etc.):

In [2]:
info_videojuegos = dict(zip(df_con_ids['item_id'], df_con_ids['Game Title']))

El dataframe final que se utilizará para las recomendaciones es:

In [3]:
df_final

,user_id,item_id,rating
0,861,12,3.670051
1,1295,38,3.862944
2,1131,21,2.695431
3,1096,4,3.873096
4,1639,14,3.030457
...,...,...,...
47769,1293,21,4.197970
47770,2485,37,2.431472
47771,2675,3,2.685279
47772,2599,37,2.258883


## 2- Definición del modelo, formateo de datos, y entrenamiento

Se formatean los datos para la librería RecBole, se define el modelo, y se entrena:

In [4]:
import os
import pandas as pd
import torch
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.model.general_recommender import LightGCN
from recbole.trainer import Trainer
from recbole.utils.case_study import full_sort_topk

#Se lee el archivo del dataset final con las ids de usuario e ítem:
df_lightgcn = pd.read_csv("video_game_reviews_with_userid_clean.csv")

#Se renombran las columnas para que RecBole las reconozca:
df_lightgcn.columns = [
    "user_id:token",
    "item_id:token",
    "rating:float"
]

#Se guarda el dataset en formato .inter y en la carpeta que RecBole requiere:
dataset_name = "video_game_reviews_recbole"
dataset_dir = os.path.join(".", dataset_name)
os.makedirs(dataset_dir, exist_ok=True)
inter_file = os.path.join(dataset_dir, f"{dataset_name}.inter")
df_lightgcn.to_csv(inter_file, sep="\t", index=False)

#Diccionario de configuración requerido por RecBole para entrenar el modelo:
config_dict = {
    "data_path": ".",
    "dataset": dataset_name,
    "dataset_file": {"inter": f"{dataset_name}/{dataset_name}.inter"},
    "field_separator": "\t",
    "USER_ID_FIELD": "user_id",
    "ITEM_ID_FIELD": "item_id",
    "RATING_FIELD": "rating",
    "load_col": {"inter": ["user_id", "item_id", "rating"]},
    "field_type": {
        "user_id": "token",
        "item_id": "token",
        "rating": "float"
    },
    "eval_args": {"split": {"RS": [0.8, 0.1, 0.1]}, "mode": "full"},
    "epochs": 10,
    "train_batch_size": 2048,
    "eval_batch_size": 4096,
    "embedding_size": 64,
    "show_progress": True,
    "learning_rate": 0.001,
    "reg_weight": 1e-5,
    "n_layers": 3,
    "topk": [10],
    "device": "cpu",
    #Umbral de rating para considerar un ítem como relevante.
    #En este caso, todo ítem con rating >= 4.0 es considerado relevante.
    "threshold": {"rating": 4.0} 
}

#Se crea el objeto Config de RecBole que guarda toda la configuración
# requerida por la librería, se carga el dataset, y se preparan los datos:
config = Config(model='LightGCN', dataset=dataset_name, config_dict=config_dict)
dataset = create_dataset(config)
train_data, valid_data, test_data = data_preparation(config, dataset)

#Se crea el modelo LightGCN y el Trainer de RecBole:
model = LightGCN(config, train_data.dataset).to(config['device'])
trainer = Trainer(config, model)

#Se entrena el modelo
trainer.fit(train_data, valid_data)
print()
print("Entrenamiento finalizado")

c:\Users\felip\AppData\Local\Programs\Python\Python311\Lib\site-packages\recbole\data\dataset\dataset.py:648: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
c:\Users\felip\AppData\Local\Programs\Python\Python311\Lib\site-packages\recbole\data\dataset\dataset.py:650: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the interm


Entrenamiento finalizado


## 3- Generación de recomendaciones

Ahora, a partir del entrenamiento anterior se obtienen las listas de recomendación top 10 para cada usuario. Se comienza usando las ids internas de la librería, para luego convertirlas a las originales del dataset:

In [5]:
#Se guardan las ids internas de los usuarios presentes en el dataset,
# evitando la id = 0 que se asocia al ['PAD'] que usa internamente RecBole
dataset_obj = test_data.dataset
uid_series = [uid for uid in range(1, dataset_obj.user_num) 
              if test_data.uid2history_item[uid] is not None]

#Se obtienen las listas de recomendación top 10 para cada usuario
topk = 10
topk_result = full_sort_topk(uid_series, model, test_data, k=topk, device=config['device'])

Se guardan en un diccionario las recomendaciones con las ids de usuario e ítem originales, convirtiendo primero las ids internas de RecBole a las originales del dataset. El diccionario tiene como llaves la id del usuario y su valor es una lista con las 10 id de ítems recomendados: 

In [6]:
#Se convierten las ids de usuarios internas de RecBole a las originales del dataset
user_original_ids = dataset_obj.id2token('user_id', uid_series)

#Se convierten las ids de ítems internas de RecBole a las originales del dataset
topk_items_original = [
    dataset_obj.id2token('item_id', topk_result.indices[i])
    for i in range(len(uid_series))
]

#Crear diccionario final de recomendaciones

#Con las ids como strings:
# recommendations_dict = {int(user_id): items for user_id, items in zip(user_original_ids, topk_items_original)}

#Con las ids como enteros:
top10_lightgcn = {int(user_id): [int(item) for item in items] for user_id, items in zip(user_original_ids, topk_items_original)}


#Ejemplo de 5 usuarios y sus recomendaciones:
print("Ejemplo de 5 usuarios y sus recomendaciones:")
for user_id, items in list(top10_lightgcn.items())[:5]:
    print(f"Usuario {user_id} → {items}")

Ejemplo de 5 usuarios y sus recomendaciones:
Usuario 861 → [21, 3, 33, 34, 31, 9, 38, 15, 19, 13]
Usuario 1295 → [16, 12, 10, 8, 29, 26, 11, 39, 30, 24]
Usuario 1131 → [35, 29, 31, 14, 17, 32, 10, 18, 16, 27]
Usuario 1096 → [35, 14, 33, 31, 9, 15, 5, 29, 10, 24]
Usuario 1639 → [22, 30, 37, 5, 32, 28, 38, 40, 39, 2]


Usuarios con recomendaciones (en total el dataset tiene 3000):

In [7]:
len(top10_lightgcn)

3000

## 4- Métricas

La ejecución del modelo LightGCN de la librería RecBole entrega de por sí ciertas métricas: Recall@K, Precision@K, MRR@K, NDCG@K, y HitScore@K. Estas métricas se obtienen primero del objeto Trainer de la librería y se calculan sobre la data de testeo. Se obtiene un tipo OrderedDict que se convierte a diccionario:

In [8]:
test_result = trainer.evaluate(test_data)
metricas_finales = dict(test_result)
print(metricas_finales)

c:\Users\felip\AppData\Local\Programs\Python\Python311\Lib\site-packages\recbole\trainer\trainer.py:583: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.loa

{'recall@10': 0.5386, 'mrr@10': 0.3397, 'ndcg@10': 0.3681, 'hit@10': 0.585, 'precision@10': 0.0648}


Ahora, de las anteriores se guardan en variables sólo las métricas relevantes (las que se utilizan en este proyecto, todas excepto MRR@K) para más adelante mostrarlas en un resumen:

In [9]:
recall_lightgcn = metricas_finales['recall@10']
precision_lightgcn = metricas_finales['precision@10']
ndcg_lightgcn = metricas_finales['ndcg@10']
hitscore_lightgcn = metricas_finales['hit@10']

print(f"Recall@10: {recall_lightgcn}")
print(f"Precision@10: {precision_lightgcn}")
print(f"NDCG@10: {ndcg_lightgcn}")
print(f"HitScore@10: {hitscore_lightgcn}")

Recall@10: 0.5386
Precision@10: 0.0648
NDCG@10: 0.3681
HitScore@10: 0.585


Otra métrica que se utilizará es el F1 Score. Se calcula y guarda en una variable:

In [10]:
f1score_lightgcn = 2 * (precision_lightgcn * recall_lightgcn) / (precision_lightgcn + recall_lightgcn)
print(f"F1 Score@10: {f1score_lightgcn:.4f}")

F1 Score@10: 0.1157


Ahora, se calcula el MAP@K. Se comienzan por definir dos funciones base, una para calcular el Average Precision at K (AP@K) para un usuario y luego otra para calcular el MAP@K como el promedio del AP@K de todos los usuarios. Finalmente, se calcula el MAP@10:

In [11]:
def ap_at_k(items_relevantes, items_recomendados, k):
    if len(items_recomendados) > k:
        items_recomendados = items_recomendados[:k]
        
    score = 0
    num_hits = 0
    for i, p in enumerate(items_recomendados):
        if p in items_relevantes and p not in items_recomendados[:i]:
            num_hits += 1
            score += (num_hits / (i + 1))
    
    if not items_relevantes:
        return 0

    return score / min(len(items_relevantes), k)

def map_at_k(df_interacciones, dict_recomendados, k, umbral_relevante):
    ap_scores = []
    for user_id, lista_recomendados in dict_recomendados.items():
        items_relevantes = set(df_interacciones[(df_interacciones["user_id"] == user_id) & (df_interacciones["rating"] >= umbral_relevante)]["item_id"])
        ap = ap_at_k(items_relevantes, lista_recomendados, k)
        ap_scores.append(ap)
    
    return sum(ap_scores) / len(ap_scores)

map_lightgcn = map_at_k(df_final, top10_lightgcn, k=10, umbral_relevante=4.0)
print(f"MAP@10: {map_lightgcn:.4f}")

MAP@10: 0.0358


Ahora, se calcula la diversidad promedio de las recomendaciones (Diversity). Para esto se analizan los géneros de videojuegos recomendados y la métrica representa cuántos géneros distintos de videojuegos se recomiendan en promedio. Se comienza definiendo dos funciones para realizar los cálculos y luego se obtiene Diversity: 

In [12]:
#Código basado en el elaborado por Nicolás Bueno y Felipe Fuentes en Tarea del curso.

#Función para calcular la cantidad de géneros distintos de videojuegos en la lista de recomendación de un usuario
def diversity_user(items_recomendados, dict_item_genero):
    if not items_recomendados:
        return 0
    
    unique_generos = set()
    for item_id in items_recomendados:
        if item_id in dict_item_genero:
            unique_generos.add(dict_item_genero[item_id])

    return float(len(unique_generos))

#Función para calcular la cantidad promedio de géneros distintos de videojuegos en todas las listas de recomendación generadas
def diversity(recomendaciones, dict_item_genero):
    total = 0
    cant_usuarios = 0
    for recs in recomendaciones.values():
        total += diversity_user(recs, dict_item_genero)
        cant_usuarios += 1
        
    return total / max(cant_usuarios, 1)

dict_item_genero = dict(zip(df_con_ids["item_id"].astype(int), df_con_ids["Genre"]))
diversity_lightgcn = diversity(top10_lightgcn, dict_item_genero)
print(f"Diversidad promedio: {diversity_lightgcn}")

Diversidad promedio: 6.728


En resumen, las métricas calculadas del modelo son:

In [13]:
print("Resumen de las métricas:")
print()
print(f"Recall@10: {recall_lightgcn:.4f}")
print(f"Precision@10: {precision_lightgcn:.4f}")
print(f"F1 Score@10: {f1score_lightgcn:.4f}")
print(f"NDCG@10: {ndcg_lightgcn:.4f}")
print(f"HitScore@10: {hitscore_lightgcn:.4f}")
print(f"MAP@10: {map_lightgcn:.4f}")
print(f"Diversity: {diversity_lightgcn:.4f}")

Resumen de las métricas:

Recall@10: 0.5386
Precision@10: 0.0648
F1 Score@10: 0.1157
NDCG@10: 0.3681
HitScore@10: 0.5850
MAP@10: 0.0358
Diversity: 6.7280


## 5- Referencias

[1] Deng, K., He, X., Li, Y., Wang, M., Wang, X., & Zhang, Y. (2020). LightGCN: Simplifying and powering graph Convolution Network for recommendation. ArXiv. Obtenido de http://arxiv.org/abs/2002.02126

[2] Diapositivas de la clase "Clase de Evaluación: metricas de error y ranking", como apoyo para implementar cálculo de métricas. Enlace: https://github.com/PUC-RecSys-Class/RecSysPUC-2025-2/blob/master/clases/s3_c1-metricas_v3.pdf

[3] Documentación de RecBole: https://github.com/RUCAIBox/RecBole?utm_source=chatgpt.com

[4] Práctico de métricas del curso del semestre pasado, utilizado como base principalmente para programar las métrica de Diversity. Enlace: https://github.com/PUC-RecSys-Class/RecSysPUC-2025-2/blob/master/practicos/pr%C3%A1ctico_m%C3%A9tricas.ipynb

[5] Uso de IA. Se consultó a ChatGPT sobre la librería RecBole, el uso de funciones y formatos de LightGCN, y errores. Link al chat: https://chatgpt.com/share/68ffb7df-f778-8008-b287-14268d81d203